In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

In [ ]:
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

In [ ]:
# Define in and out destinations

root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/Code/MitoticStop/" + "/"
outputFolder = root + "Example Output" + "/"

In [ ]:
# Define time interval between frames in minutes
frame_interval = 7

# Define max frame for lineage tracing 
cutoff_frame = 823

In [ ]:
def lineages(x, links_dataframe):
    # Parse lineage relationships between splitting events
    source_id = x.Source_ID # each row's source spot ID
    # add source id to list if the row is a splitting event
    if x.Splitting_event:
        lineage = [str(source_id)]
    else:
        lineage = []
    while True:
        target_id = links_dataframe.loc[links_dataframe['Target_ID'] == source_id,:] # find row containing corresponding target ID
        if target_id.empty:
            break
        if target_id.Splitting_event.values[0]:
            lineage.append(str(target_id.Source_ID.values[0]))
        source_id = target_id.Source_ID.values[0]
    if lineage:    
        return ".".join(reversed(lineage)) # insert . between every element in the list, in reverse order
    else:
        return None

def get_mother(x):
    lineage_list = x.split(".")
    if len(lineage_list) == 1:
        return None
    else:
        return lineage_list[-2] 

def get_grandmother(x):
    lineage_list = x.split(".")
    if len(lineage_list) < 3:
        return None
    else:
        return lineage_list[-3]     
    
def get_sister(x, dataframe):
    if x.Mother_ID is None:
        return None
    else:
        sister_df = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Mother_ID == x.Mother_ID) & 
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID != x.Source_ID), :
        ]
        if sister_df.shape[0] > 0:
            lineage_sister = sister_df.iloc[0]['Lineage']
            sister = lineage_sister.split(".")[-1]
            return sister
        else:
            pass

def get_daughters(x, dataframe):
    daughters_df = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Track_ID == x.Track_ID) &
            (dataframe.Mother_ID == x.Source_ID) & 
            (dataframe.Position == x.Position), :
    ]
    daughters_list = daughters_df.Source_ID.unique()
    return daughters_list

def get_mother_n_of_daughters(x, dataframe):

    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID) &
            (dataframe.Position == x.Position)
        ]
        
        n_daughters = mother_row.Number_of_daughters_in_mitosis.item()
        return n_daughters

def seensister(x, already_seen_list):
    if x.Sister_ID is None:
            return None
    else:
        if x.Mother_ID in already_seen_list:
            return True
        else:
            already_seen_list.append(x.Mother_ID)
            return False

def get_cousin(x, dataframe):
    if x.Grandmother_ID is None:
        return None
    else:
        cousins_df = dataframe.loc[
            (dataframe.Grandmother_ID == x.Grandmother_ID) & 
            (dataframe.Source_ID != x.Source_ID) &
            (dataframe.Mother_ID != x.Mother_ID), :
        ]
        cousins = []
        if cousins_df.shape[0] == 2:   
            cousinA = cousins_df.iloc[0, 1].item()
            cousinB = cousins_df.iloc[1, 1].item()
            return cousinA, cousinB
        elif cousins_df.shape[0] == 1:
            cousinA = cousins_df.iloc[0, 1].item()
            return cousinA
        else:
            return None

def get_random(x, dataframe):
    random_df = dataframe.loc[
            (dataframe.Dataset == x.Position) &
            (dataframe.Position == x.Position) &
            (dataframe.Track_ID != x.Track_ID) & # Don't pair within same lineage
            (dataframe.Source_ID != x.Sister_ID) & 
            (dataframe.Source_ID != x.Mother_ID)  
           # (dataframe.Source_ID != x.Grandmother_ID) & 
           # (dataframe.Source_ID != x.Cousin_ID) & 
           # (dataframe.Source_ID != x.Aunt_ID), :
        ]
    
    random_ID = random_df["Source_ID"].sample(1)
    return random_ID.item()

def seengranny(x, granny_seen_list):
    if x.Grandmother_ID is None:
        return None
    else:
        if x.Grandmother_ID not in granny_seen_list:
            granny_seen_list.append(x.Grandmother_ID)
            return False
        else:
            return True

def get_cell_cycle(x, dataframe):
    daughter_time = x.Frame
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        if mother_row["Frame"].shape[0] == 1:   
            mother_time = mother_row["Frame"].item()
            cell_cycle = (int(daughter_time) - int(mother_time)) * frame_interval
            return cell_cycle
        else:
            return None


          
def tracking_dataframes(edges_dir, spots_dir, dataset, position, condition, outdir, max_frame = cutoff_frame):
    links = pd.read_csv(
        edges_dir,
        usecols = ['TRACK_ID', 'SPOT_SOURCE_ID', 'SPOT_TARGET_ID']
    )
    links.rename(columns = {
        "TRACK_ID": "Track_ID", 
        "SPOT_SOURCE_ID": "Source_ID", 
        "SPOT_TARGET_ID": "Target_ID"
    }, inplace = True)
    
    spots = pd.read_csv(
        spots_dir,
        usecols = ['ID', 'TRACK_ID', 'POSITION_X', 'POSITION_Y', 'FRAME']
    )
    spots.rename(columns = {
        "TRACK_ID": "Track_ID", 
        "POSITION_X": "Track_Coordinate_X", 
        "POSITION_Y": "Track_Coordinate_Y", 
        "FRAME": "Frame"
    }, inplace = True)

    # Crop spots by frame
    spots["Frame"] = pd.to_numeric(spots["Frame"], errors = "coerce")
    spots = spots[spots["Frame"] < max_frame]

    # Crop links based on remaining spots
    valid_ids = set(spots["ID"])

    links = links[
        links["Source_ID"].isin(valid_ids) &
        links["Target_ID"].isin(valid_ids)
    ]
    
    # Find spots that don't have a target and are not at the end of the movie

    # Find all end-point IDs = Target_IDs that never appear as Source_IDs (per Track_ID)
    end_points = (
        links
        .loc[~links['Target_ID'].isin(links['Source_ID'])]
        .groupby('Track_ID')['Target_ID']
        .unique()
        .reset_index(name = 'EndPoint_ID')
    )

    spots_with_endpoints = spots.merge(
        end_points.explode('EndPoint_ID'),
        left_on = ['Track_ID', 'ID'],
        right_on = ['Track_ID', 'EndPoint_ID'],
        how = 'inner'
    )

    last_frame = spots_with_endpoints.Frame.max()
    spots_with_endpoints = spots_with_endpoints[spots_with_endpoints["Frame"] < last_frame]
    
    # Find all downstream spots of each endpoint
    
    from collections import defaultdict
    
    downstream_mapping = []  # will hold {Track_ID, EndPoint_ID, Downstream_IDs}
    
    for track_id, group in links.groupby("Track_ID"):
        # Build adjacency list (Source -> Target)
        adjacency = defaultdict(list)
        for _, row in group.iterrows():
            adjacency[row["Source_ID"]].append(row["Target_ID"])
    
        # Build reverse mapping (Target -> Source) to go upstream
        reverse_adj = defaultdict(list)
        for src, targets in adjacency.items():
            for tgt in targets:
                reverse_adj[tgt].append(src)
    
        # Find endpoints for this track
        endpoints = group.loc[~group["Target_ID"].isin(group["Source_ID"]), "Target_ID"].unique()

        for endpoint in endpoints:
            visited = set()
            stack = [endpoint]
        
            while stack:
                current = stack.pop()
                if current in visited:
                    continue
        
                visited.add(current)
        
                # move upstream
                parents = reverse_adj[current]
        
                for parent in parents:
                    # STOP if parent is a split node
                    if len(adjacency[parent]) > 1:
                        visited.add(parent)
                        continue
        
                    stack.append(parent)
        
            downstream_mapping.append({
                "Track_ID": track_id,
                "EndPoint_ID": endpoint,
                "Downstream_IDs": list(visited)
            })

    # Convert to DataFrame
    downstream_df = pd.DataFrame(downstream_mapping)

    # Merge with filtered endpoints
    downstream_df = spots_with_endpoints.merge(downstream_df) 

    destination_downstream = outdir + "_downstream.csv"
    downstream_df.to_csv(destination_downstream)
    print("Successfully exported endpoint dataframe to " + destination_downstream)

    # Generate a list of spot_ids that correspond to a splitting event
    # (Per definition, a splitting event is labelled twice as "source")
    source_ids = list(links["Source_ID"])
    source_id_counts = Counter(source_ids)
    splitting_event_ids = [id for id in source_id_counts if source_id_counts[id] > 1]
    
    # Add Boolean to Spots and Links dataframes
    # that indicate whether the spot or links belongs to a 
    # splitting event

    spots["Splitting_event"] = spots["ID"].apply(lambda x:\
                                                 False if x not in splitting_event_ids\
                                                 else True)

    links["Splitting_event"] = links["Source_ID"].apply(lambda x:\
                                                 False if x not in splitting_event_ids\
                                                 else True)
    
    links['Lineage'] = links.apply(lineages, args = (links,), axis = 1)
    print("Successfully annotated lineage information.")

    links = links[links["Splitting_event"] == True]
    spots = spots[spots["Splitting_event"] == True]
    spots.rename(columns = {"ID": "Source_ID"}, inplace = True)
        
    df = pd.merge(links, spots, how = "outer", on = ["Source_ID", "Track_ID", "Splitting_event"])
   
    df.drop(["Target_ID"], axis = 1, inplace = True)
    df.drop_duplicates(subset = ['Source_ID'], inplace = True)

    downstream_df.drop(["Track_Coordinate_X"], axis = 1, inplace = True)
    downstream_df.drop(["Track_Coordinate_Y"], axis = 1, inplace = True)
    downstream_df.drop(["Frame"], axis = 1, inplace = True)
    # First, explode downstream_df so each Downstream_ID is its own row
    downstream_exploded = downstream_df.explode("Downstream_IDs")
    
    # Merge df with downstream_exploded on Source_ID == Downstream_IDs
    df_with_endpoints = df.merge(
        downstream_exploded,
        left_on = "Source_ID",
        right_on = "Downstream_IDs",
        how = "left"
    )
    
    # If df also has Track_ID, join on both Track_ID and Source_ID to be safe:
    df_with_endpoints = df.merge(
         downstream_exploded,
         left_on = ["Track_ID", "Source_ID"],
         right_on = ["Track_ID", "Downstream_IDs"],
         how = "left"
     )
    
    # If multiple EndPoint_IDs match a given Source_ID, group them into a list
    df_with_endpoints = (
        df_with_endpoints.groupby(df.columns.tolist(), dropna = False)["EndPoint_ID"]
        .agg(lambda x: list(x.dropna()))
        .reset_index()
    )

    destination_splittingEvents_downstream = outdir + "_SplittingEventsDownstream.csv"
    df_with_endpoints.to_csv(destination_splittingEvents_downstream)
    print("Successfully exported endpoint dataframe to " + destination_splittingEvents_downstream)

    df = df.merge(df_with_endpoints)
    
    df["Position"] = position    
    df["Dataset"] = dataset
    df["Condition"] = condition
  
    df["Generation"] = df["Lineage"].apply(lambda x: x.count(".") + 1)
    df["Mother_ID"] = df["Lineage"].apply(lambda x: get_mother(x))
    df["Grandmother_ID"] = df["Lineage"].apply(lambda x: get_grandmother(x))
    df["Sister_ID"] = df.apply(get_sister, dataframe = df, axis = 1)
    df["Daughters_ID"] = df.apply(get_daughters, dataframe = df, axis = 1)
    df["Number_of_daughters_in_mitosis"] = df['Daughters_ID'].str.len()
    df["Mother_Number_of_daughters_in_mitosis"] = df.apply(get_mother_n_of_daughters, dataframe = df, axis = 1)
    
    df["Cell_Cycle_mins"] = df.apply(get_cell_cycle, dataframe = df, axis = 1)
    df["Cell_Cycle_hours"] = df.Cell_Cycle_mins / 60

    destination = outdir + "_lineages.csv"
    df.to_csv(destination)
    print("Successfully exported lineage dataframe to " + destination)
    
    return df

In [ ]:
# define input dataframes (in this example, one for the control siLUC1 and one for augmin depletion siHAUS6)

df_01 = tracking_dataframes(
    edges_dir = root + 'Example Data/1_TrackMate_edges.csv', 
    spots_dir = root + 'Example Data/1_TrackMate_spots.csv',
    dataset = "Example",
    position = 1, 
    condition = "siLUC1",
    outdir = outputFolder + "1_TrackMate"
)

df_02 = tracking_dataframes(
    edges_dir = root + 'Example Data/2_TrackMate_edges.csv', 
    spots_dir = root + 'Example Data/2_TrackMate_spots.csv',
    dataset = "Example",
    position = 2,
    condition = "siHAUS6",
    outdir = outputFolder + "2_TrackMate"
)

In [ ]:
# Export example output

# Collect, concatenate and save dataframe

def collect_dataframes(start, end, prefix = "df_"):
    dfs = {}
    for i in range(start, end + 1):
        name_padded = f"{prefix}{i:02d}"
        name_plain = f"{prefix}{i}"

        if name_padded in globals():
            dfs[i] = globals()[name_padded]
        elif name_plain in globals():
            dfs[i] = globals()[name_plain]

    return dfs
dataframes = collect_dataframes(1, 2) # Specify dataframes to include
print(sorted(dataframes.keys()))

df_concat = pd.concat(dataframes.values(), ignore_index = True)
df_concat.to_csv(outputFolder + "MainDataFrame_Lineages_Example.csv")

print("Successfully exported Main lineages dataframe.")

In [ ]:
# Summary of lineage statistics

from statistics import mean

max_frame = cutoff_frame

def get_lineage_statistics(max_frame = max_frame, dataframe = dataframes, outDir = outputFolder):
    
    dataframes_for_concat = []
        
    for data in dataframe.values():
        data["Frame"] = data.Frame.astype(int)

        data = data[data["Frame"] < max_frame]
        

        first_row = data.head(1)
        dataset = first_row["Dataset"].iloc[0]
        position = first_row["Position"].iloc[0]
        condition = first_row["Condition"].iloc[0]

        max_generation = data.Generation.max()
        No_of_source_ids = data.Source_ID.dropna().shape[0]
        No_of_sister_ids = data.Sister_ID.dropna().shape[0]
        No_of_mother_ids = data.Mother_ID.dropna().shape[0]
        No_of_grandmother_ids = data.Grandmother_ID.dropna().shape[0]
        
        list_of_tracks = data.Track_ID.unique()
        list_of_trackSubData = []
        for track in list_of_tracks:
            sub_data = data.loc[data.Track_ID == track]
            list_of_trackSubData.append(sub_data)

        No_of_tracks = data.Track_ID.nunique()
        Average_No_of_splitting_events = data.groupby("Track_ID").describe().count().mean()

        sister_per_split = []
        mother_per_split = []
        grandmother_per_split = []

        for sub_data in list_of_trackSubData: 
            number_of_sister_ids = sub_data.Sister_ID.dropna().shape[0]
            sis_per_total_splits = number_of_sister_ids / sub_data.shape[0]
            sister_per_split.append(sis_per_total_splits)

            number_of_mother_ids = sub_data.Mother_ID.dropna().shape[0]
            mom_per_total_splits = number_of_mother_ids / sub_data.shape[0]
            mother_per_split.append(mom_per_total_splits)

            number_of_gmother_ids = sub_data.Grandmother_ID.dropna().shape[0]
            gmom_per_total_splits = number_of_gmother_ids / sub_data.shape[0]
            grandmother_per_split.append(gmom_per_total_splits)

        statistics_dict = {"Dataset": dataset, 
                           "Position": position,
                           "Condition": condition,
                           "Number_of_tracks": No_of_tracks,
                           "Max_No_of_Generations": max_generation,
                           "Number_of_Source_IDs": No_of_source_ids,
                           "Number_of_Sister_IDs": No_of_sister_ids,
                           "Number_of_Mother_IDs": No_of_mother_ids,
                           "Number_of_Grandmother_IDs": No_of_grandmother_ids,
                           "Average_No_of_SplittingEvents_per_Track": data.shape[0] / No_of_tracks, 
                           "Average_No_of_Sisters_per_Splitting_event": mean(sister_per_split), 
                           "Average_No_of_Mothers_per_Splitting_event": mean(mother_per_split), 
                           "Average_No_of_Grandmothers_per_Splitting_event": mean(grandmother_per_split), 
                          }
        df = pd.DataFrame(statistics_dict, index = [0])
        dataframes_for_concat.append(df)

    final_df = pd.concat(dataframes_for_concat)
    
    final_df.to_csv(outDir + "MetaStatistics_TrackingLineages_Example.csv")
    return final_df

statistics_df = get_lineage_statistics()
statistics_df